[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C09_Reasoning_TTC_Course/05_ttc_scaling/05_ttc_scaling_laws.ipynb)

# 05 · 测试时计算 Scaling Laws：算力怎么花最划算

<span style="background:#1f6feb;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span> 纯 numpy/matplotlib 模拟，自包含，所有 cell 秒级跑完。

**本 notebook 你将完成：**

1. 构造**难度分层**的合成题目集：难度 $d$ 服从截断正态分布，单样本正确率 $q(d, M)$ 由模型规模 $M$ 与难度共同决定；
2. 对每个难度桶画 **accuracy vs N**（采样数）曲线，复现 [Snell 2024] 的难度依赖结论：简单题快速饱和、中等题持续受益、超难题 TTC 救不了；
3. 实现 **compute-optimal TTC 分配器**（贪心边际收益），验证按难度自适应分配严格优于均匀分配；
4. 算清 **FLOPs 账本**（一次前向 $\approx 2N_{\text{params}}\cdot$tokens），画"小模型+多采样 vs 大模型+少采样"的 **iso-accuracy 等高线**并找出交叉区域；
5. 手写 **log-log 幂律拟合**（`np.polyfit`），复现 [Brown 2024] 的聚合 coverage 近似幂律；
6. 4 道 ✏️ 练习巩固核心函数。

参考：[Brown 2024] *Large Language Monkeys* (arXiv:2407.21787)、[Snell 2024] *Scaling LLM Test-Time Compute Optimally can be More Effective than Scaling Model Parameters* (arXiv:2408.03314)、[Wu 2024] *Inference Scaling Laws* (arXiv:2408.00724)、[Muennighoff 2025] *s1: Simple test-time scaling* (arXiv:2501.19393)。

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams["font.sans-serif"] = ["PingFang SC", "Hiragino Sans GB", "Noto Sans CJK SC", "Arial Unicode MS", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
rng = np.random.default_rng(0)

# ============ 玩具世界的"物理定律" ============
# 单样本正确率 q(d, M)：难度 d 与模型规模 M 在 logit 尺度上对决
#     logit q = A_PARAM * (log10(M) - 7) - B_DIFF * d + C0
# A_PARAM = 每 10 倍参数买到的 log-odds（"参数的汇率"）。
# 对照：q 很小时，每 10 倍采样数 N 买到 ln(10) ≈ 2.3 的有效 log-odds（coverage ≈ qN）。
# 我们故意取 A_PARAM = 1.8 < 2.3，让"参数 vs 采样"存在真实权衡（第 3 节的交叉区域）；
# 真实世界里这两个汇率孰大孰小正是 Snell/Wu 的实验要测的量。
A_PARAM, B_DIFF, C0 = 1.8, 2.0, 4.0

def q_single(d, M):
    """单样本正确率 q(d, M)。d 可为标量或数组，M 为参数量。"""
    logit = A_PARAM * (np.log10(M) - 7.0) - B_DIFF * np.asarray(d, dtype=float) + C0
    return 1.0 / (1.0 + np.exp(-logit))

# ============ 难度分层的题目集：300 题，d ~ N(4.5, 2.2) 截断到 [0, 10] ============
N_PROBLEMS = 300
difficulties = np.clip(rng.normal(4.5, 2.2, size=N_PROBLEMS), 0.0, 10.0)

BINS = [(0, 2, "很容易"), (2, 4, "容易"), (4, 6, "中等"), (6, 8, "难"), (8, 10.01, "超难")]
M_BASE = 3e8   # 基线小模型：0.3B 参数

print(f"{'难度桶':<4} {'题数':>4} {'平均 d':>7} {'平均 q(d, 0.3B)':>16}")
for lo, hi, name in BINS:
    mask = (difficulties >= lo) & (difficulties < hi)
    qs = q_single(difficulties[mask], M_BASE)
    print(f"{name:<4} {mask.sum():>4} {difficulties[mask].mean():>7.2f} {qs.mean():>16.2e}")

## 1 · accuracy vs N：复现 Snell 的难度依赖结论

对一道单样本正确率为 $q$ 的题独立采 $N$ 条样本，在 **oracle verifier**（能从 $N$ 条中认出正确答案，即 pass@N）下：

$$\text{coverage}(q, N) = 1 - (1-q)^N$$

对每个难度桶取平均，画 coverage vs $N$。预期看到 [Snell 2024] 的三段结构：**简单题快速饱和 / 中等题持续受益 / $q \approx 0$ 的超难题 $N$ 再大也没用**。

In [ ]:
Ns = 2 ** np.arange(0, 11)          # N = 1 .. 1024

def coverage_curve(qs, Ns):
    """题目子集 qs 在各采样数 N 下的平均 coverage（oracle verifier = pass@N）。"""
    qs = np.asarray(qs, dtype=float)
    return np.array([np.mean(1.0 - (1.0 - qs) ** N) for N in Ns])

plt.figure(figsize=(7.5, 4.5))
for lo, hi, name in BINS:
    mask = (difficulties >= lo) & (difficulties < hi)
    qs = q_single(difficulties[mask], M_BASE)
    plt.plot(Ns, coverage_curve(qs, Ns), marker="o", label=f"{name} (q≈{qs.mean():.1e})")
plt.xscale("log", base=2)
plt.xlabel("N (samples per problem)")
plt.ylabel("coverage = pass@N")
plt.title("coverage vs N by difficulty bin (M = 0.3B)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print(f"{'难度桶':<4} {'N=1':>7} {'N=16':>7} {'N=1024':>8}   {'16→1024 增益':>10}")
for lo, hi, name in BINS:
    mask = (difficulties >= lo) & (difficulties < hi)
    qs = q_single(difficulties[mask], M_BASE)
    c1, c16, c1024 = coverage_curve(qs, [1, 16, 1024])
    print(f"{name:<4} {c1:>7.3f} {c16:>7.3f} {c1024:>8.3f}   {c1024 - c16:>+10.3f}")

**读图（对照 Snell 2024 的难度分桶结论）：**

- **很容易 / 容易**：$N=4$ 左右 coverage 就贴近 1.0，继续加采样是纯浪费；
- **中等**：从 $N=16$ 到 $N=1024$ 仍有约 +0.5 的绝对提升——TTC 花在这里最划算；
- **难**：$q \sim 10^{-3}$，要 $N \approx 1/q$ 量级才起飞，$N=1024$ 也只到 ~0.6；
- **超难**：$q \sim 10^{-5}$，$N=1024$ 时 coverage ≈ 0.03——**TTC 救不了，只能加训练算力**。

## 2 · compute-optimal TTC 分配器：贪心边际收益

给定总预算 $B$ 条样本，怎么分给 300 道题？第 $i$ 题已有 $n_i$ 条样本时，**再给一条的边际收益**（期望解出题数的增量）：

$$\Delta_i(n_i) = \big[1-(1-q_i)^{n_i+1}\big] - \big[1-(1-q_i)^{n_i}\big] = q_i\,(1-q_i)^{n_i}$$

$\Delta_i$ 随 $n_i$ **严格递减**（目标函数离散凹），所以"每次把下一条样本给当前 $\Delta$ 最大的题"的贪心就是全局最优。直觉上它会：给简单题 1–2 条（够用就停）、把大头砸给中等题（边际收益持续高）、给超难题 0 条（第一条的边际收益就低于别处的机会成本）。

In [ ]:
import heapq

qs_all = q_single(difficulties, M_BASE)

def expected_solved(qs, alloc):
    """期望解出题数 sum_i [1 - (1-q_i)^{n_i}]。"""
    return float(np.sum(1.0 - (1.0 - np.asarray(qs, dtype=float)) ** np.asarray(alloc)))

def greedy_allocation(qs, B):
    """贪心边际收益分配（堆实现，O(B log n)）。"""
    qs = np.asarray(qs, dtype=float)
    alloc = np.zeros(len(qs), dtype=int)
    heap = [(-q, i) for i, q in enumerate(qs)]      # (-边际收益, 题号)
    heapq.heapify(heap)
    for _ in range(B):
        neg_gain, i = heapq.heappop(heap)
        alloc[i] += 1
        heapq.heappush(heap, (neg_gain * (1.0 - qs[i]), i))   # Δ_i ← Δ_i · (1 - q_i)
    return alloc

B = 4 * N_PROBLEMS                                   # 总预算：平均每题 4 条
uniform  = np.full(N_PROBLEMS, B // N_PROBLEMS)
adaptive = greedy_allocation(qs_all, B)
assert adaptive.sum() == B                           # 预算约束

ev_uni = expected_solved(qs_all, uniform)
ev_ada = expected_solved(qs_all, adaptive)
print(f"总预算 B = {B} 条样本")
print(f"均匀分配（每题 4 条）期望解出 {ev_uni:6.1f} / {N_PROBLEMS}")
print(f"自适应贪心分配        期望解出 {ev_ada:6.1f} / {N_PROBLEMS}  (+{(ev_ada / ev_uni - 1) * 100:.1f}%)")
assert ev_ada > ev_uni, "自适应分配必须严格优于均匀分配"

print(f"\n{'难度桶':<4} {'平均分到的样本数':>14}")
for lo, hi, name in BINS:
    mask = (difficulties >= lo) & (difficulties < hi)
    print(f"{name:<4} {adaptive[mask].mean():>14.2f}")

**注意**：难题和超难题分到 **0 条**——预算全部流向"很容易补 1–2 条保险 + 容易/中等吃大头"。这正是 [Snell 2024] "compute-optimal 策略比固定 best-of-N 省 ~4x 算力"的机制（玩具世界里增益约 +15%，幅度取决于难度分布与预算规模）。也要看到前提：**分配器知道每题的 oracle $q_i$**。实践中 $q_i$ 要靠先采几条来估（model-predicted difficulty），这笔探路成本会吃掉一部分收益。

## 3 · FLOPs 账本与 iso-accuracy 等高线

统一的算力度量（dense Transformer 标准近似）：

$$\text{一次前向} \approx 2 N_{\text{params}} \cdot n_{\text{tokens}}, \qquad \text{训练} \approx 6 N_{\text{params}} \cdot D_{\text{tokens}}$$

先算一笔经典账："训练 10 倍大的模型" vs "小模型每题采 100 条"；再在 $(\log_{10} M,\ \log_2 N)$ 平面上画 **iso-accuracy 等高线**（蓝）与 **iso-FLOPs 线**（灰虚线，$M \cdot N = \text{const}$），每条等高线与灰线的切点就是该 accuracy 目标下的 compute-optimal 组合。

In [ ]:
L_GEN = 1000                      # 每条样本平均生成 1000 tokens（含 CoT）

def infer_flops(M, N_samples, L=L_GEN):
    """每题推理 FLOPs ≈ 2 · M · L · N。"""
    return 2.0 * M * L * N_samples

# ---- 账本：训练大 10 倍 vs 推理采 100 条 ----
M_small, M_big = 7e9, 70e9
train_big_cost = 6.0 * M_big * (20.0 * M_big)        # Chinchilla 配比 D = 20·N_params
cost_small_100 = infer_flops(M_small, 100)
cost_big_1     = infer_flops(M_big, 1)
print(f"训练 70B（D=20N）一次性成本 : {train_big_cost:.2e} FLOPs")
print(f"7B  每题 100 条样本         : {cost_small_100:.2e} FLOPs/题")
print(f"70B 每题 1 条样本           : {cost_big_1:.2e} FLOPs/题")
breakeven = train_big_cost / (cost_small_100 - cost_big_1)
print(f"盈亏平衡请求量 ≈ {breakeven:.1e} 题：低于它，'7B+采样' 总账更便宜（R = 推理/训练 token 比，越小越偏向 TTC）\n")

# ---- iso-accuracy 等高线 ----
Ms_grid = np.logspace(8, 11, 31)        # 0.1B – 100B
Ns_grid = 2.0 ** np.arange(0, 11)       # 1 – 1024
acc_grid = np.zeros((len(Ns_grid), len(Ms_grid)))
for j, M in enumerate(Ms_grid):
    qd = q_single(difficulties, M)
    for i, n in enumerate(Ns_grid):
        acc_grid[i, j] = np.mean(1.0 - (1.0 - qd) ** n)

X, Y = np.meshgrid(np.log10(Ms_grid), np.log2(Ns_grid))
plt.figure(figsize=(8, 5))
cs = plt.contour(X, Y, acc_grid, levels=[0.3, 0.5, 0.7, 0.8, 0.9], colors="C0")
plt.clabel(cs, fmt="%.1f")
for c in [1e13, 1e14, 1e15]:            # iso-FLOPs：M·N = const，该平面上的直线
    plt.plot(np.log10(Ms_grid), np.log2(c / (2.0 * L_GEN * Ms_grid)), "--", color="gray", lw=1)
plt.ylim(0, 10)
plt.xlabel("log10(M params)"); plt.ylabel("log2(N samples)")
plt.title("iso-accuracy (blue) vs iso-FLOPs (gray dashes)")
plt.tight_layout(); plt.show()

# ---- 每个 accuracy 目标的最省 FLOPs 组合 vs 只许 N=1 ----
for target in [0.5, 0.7, 0.9]:
    feas = [(infer_flops(Ms_grid[j], Ns_grid[i]), Ms_grid[j], Ns_grid[i])
            for i in range(len(Ns_grid)) for j in range(len(Ms_grid)) if acc_grid[i, j] >= target]
    f, M, n = min(feas)
    combo = f"M={M:.1e}, N={int(n):>4}, {f:.1e} FLOPs/题"
    n1 = [(infer_flops(Ms_grid[j], 1), Ms_grid[j]) for j in range(len(Ms_grid)) if acc_grid[0, j] >= target]
    n1_str = f"M={min(n1)[1]:.1e}, {min(n1)[0]:.1e} FLOPs/题" if n1 else "网格内无解（多大的模型一条都不够）"
    print(f"目标 acc≥{target}: 最省组合 [{combo}]   只许 N=1 [{n1_str}]")

**读图**：目标 acc=0.5 时最省组合是 ~0.16B 模型 + $N=8$，比"最小的能单发达标的模型"（~5B）便宜约 4 倍——**小模型+多采样赢**；acc=0.7、0.9 时网格里没有任何模型能 $N=1$ 达标，且最省组合的 $M$ 被迫右移——**高目标下参数不可替代**。这复现了 [Snell 2024] / [Wu 2024] 的交换比结论。注意整张图的形状由 `A_PARAM`(=1.8) 与 $\ln 10 \approx 2.3$ 的相对大小决定——把 `A_PARAM` 改成 3.0 重画一遍，交叉区域就会消失（参数全面碾压采样），这正是真实实验需要测量的量。

## 4 · 聚合 coverage 的近似幂律（Brown 2024）+ 手写 log-log 拟合

单题 coverage $1-(1-q)^N$ 是饱和的 S 曲线，**不是**幂律；但对重尾分布的 $q_i$ 取平均后，log-log 图上会出现一段近似直线——$N$ 每翻一倍，就有一批 $q_i \sim 1/N$ 量级的题进入"可解窗口"。下面画出两者的对比，并用 `np.polyfit` 在 log 域拟合幂律段斜率（[Brown 2024] 用的是 exponentiated power law $c \approx \exp(a k^b)$，纯幂律是其局部近似）。

In [ ]:
Ns_pl = np.unique(np.round(np.logspace(0, 4, 25)).astype(int))
cov_agg = np.array([np.mean(1.0 - (1.0 - qs_all) ** n) for n in Ns_pl])

# 取远离两端饱和的中段，拟合 coverage ≈ exp(ln_c) · N^slope
seg_mask = (cov_agg > 0.3) & (cov_agg < 0.85)
slope, ln_c = np.polyfit(np.log(Ns_pl[seg_mask]), np.log(cov_agg[seg_mask]), 1)
fit = np.exp(ln_c) * Ns_pl.astype(float) ** slope
rel_err = np.abs(fit[seg_mask] - cov_agg[seg_mask]) / cov_agg[seg_mask]

plt.figure(figsize=(7.5, 4.5))
plt.loglog(Ns_pl, cov_agg, "o-", label="aggregate coverage (300 problems)")
plt.loglog(Ns_pl[seg_mask], fit[seg_mask], "k--", lw=2, label=f"power-law fit, slope={slope:.2f}")
for q in [0.6, 0.03, 1e-4]:             # 三条单题曲线：全部饱和，没有幂律
    plt.loglog(Ns_pl, 1.0 - (1.0 - q) ** Ns_pl.astype(float), ":", alpha=0.7, label=f"single problem q={q:g}")
plt.xlabel("N (log)"); plt.ylabel("coverage (log)")
plt.title("aggregate coverage ~ power law; single problems saturate")
plt.legend(fontsize=8); plt.grid(alpha=0.3, which="both"); plt.tight_layout(); plt.show()

print(f"幂律段拟合：slope = {slope:.3f}（每 10 倍采样，coverage × {10**slope:.2f}），段内最大相对误差 {rel_err.max():.1%}")
assert 0.0 < slope < 1.0

---
## ✏️ 练习 1：coverage 与幂律近似的相对误差

实现 `coverage_exact(q, N)`（精确 coverage $1-(1-q)^N$）和 `coverage_relerr(q, N)`（线性近似 $qN$ 相对精确值的误差 $(qN - \text{exact})/\text{exact}$）。

线性近似就是"斜率为 1 的幂律"：$qN \ll 1$ 时 $1-(1-q)^N \approx qN$，这是聚合幂律的原材料；接近饱和时它会严重高估（甚至超过 1）。

**提示**：每个函数 3 行以内；union bound 保证 $qN \ge 1-(1-q)^N$，所以相对误差恒非负；自测不会喂 $q=0$ 给 `coverage_relerr`（除零），但会喂给 `coverage_exact`。

In [ ]:
def coverage_exact(q, N):
    # TODO: 返回精确 coverage 1 - (1-q)^N
    raise NotImplementedError

def coverage_relerr(q, N):
    # TODO: 线性近似 approx = q * N，返回相对误差 (approx - exact) / exact
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert abs(coverage_exact(0.5, 1) - 0.5) < 1e-12
assert abs(coverage_exact(1.0, 3) - 1.0) < 1e-12          # q=1：一条就中
assert abs(coverage_exact(0.0, 100) - 0.0) < 1e-12        # q=0：永远不中
assert coverage_relerr(0.001, 10) < 0.01                  # qN = 0.01 << 1：近似很准
assert coverage_relerr(0.5, 10) > 1.0                     # 饱和区：近似严重高估
assert coverage_relerr(0.01, 2) >= 0.0                    # union bound ⇒ 误差非负
print("✅ 练习 1 通过")

## ✏️ 练习 2：实现 `optimal_allocation(qs, B)`

不翻上文，自己实现一遍贪心边际收益分配：输入每题单样本正确率数组 `qs` 与总预算 `B`（样本总条数），返回整数数组 `alloc`，最大化期望解出题数 $\sum_i \big[1-(1-q_i)^{n_i}\big]$，约束 $\sum_i n_i = B$、$n_i \ge 0$。

**提示**：边际收益 $\Delta_i(n) = q_i(1-q_i)^n$ 随 $n$ 严格递减 ⇒ 每次把下一条样本给当前 $\Delta$ 最大的题即全局最优。维护 `gains = qs.copy()`，分配一条后 `gains[i] *= (1 - qs[i])`，用 `np.argmax` 选择即可（$O(B \cdot n)$，自测规模很小），约 8 行。边界：$q_i = 0$ 的题边际收益恒为 0，只要还有 $q_j > 0$ 的题就永远轮不到它。

In [ ]:
def optimal_allocation(qs, B):
    # TODO: 贪心边际收益分配
    #   1) alloc = 全零整数数组；gains = qs 的拷贝（下一条样本的边际收益）
    #   2) 重复 B 次：i = argmax(gains)；alloc[i] += 1；gains[i] *= (1 - qs[i])
    #   3) 返回 alloc（sum(alloc) == B）
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
qs_t = np.array([0.0, 0.05, 0.5, 0.9])
al = np.asarray(optimal_allocation(qs_t, 20))
assert al.sum() == 20 and (al >= 0).all()                                # 预算约束
assert al[0] == 0                                                        # q=0 的题一条都不该拿
assert expected_solved(qs_t, al) > expected_solved(qs_t, [5, 5, 5, 5])   # 严格优于均匀
al2 = np.asarray(optimal_allocation(qs_all, 600))
assert al2.sum() == 600
assert expected_solved(qs_all, al2) >= expected_solved(qs_all, np.full(N_PROBLEMS, 2))  # 不劣于均匀
print("✅ 练习 2 通过")

## ✏️ 练习 3：实现 `iso_accuracy_tradeoff(target_acc, Ms, Ns, ds)`

求"达到目标平均 accuracy 的最省 FLOPs 组合"：在候选模型规模 `Ms` × 候选采样数 `Ns` 的网格上，找出平均 accuracy（$\mathrm{mean}_d\big[1-(1-q(d,M))^N\big]$，用全局的 `q_single`）$\ge$ `target_acc` 且每题推理 FLOPs（用全局的 `infer_flops(M, N)`）**最小**的组合，返回三元组 `(M, N, flops)`；全网格都不可行时返回 `None`。

**提示**：双重循环 + 收集 feasible 列表 + `min`，10 行以内；先判 feasible 是否为空再取 `min`；FLOPs 并列时任取其一。

In [ ]:
def iso_accuracy_tradeoff(target_acc, Ms, Ns, ds):
    # TODO: 在 Ms × Ns 网格上找 平均 accuracy >= target_acc 且 infer_flops(M, N) 最小的组合
    #   acc(M, N) = np.mean(1 - (1 - q_single(ds, M)) ** N)
    #   返回 (M, N, flops)；全部不可行时返回 None
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
ds_t = difficulties[:100]
Ms_t = np.logspace(8, 11, 10)
Ns_t = 2.0 ** np.arange(0, 9)
res = iso_accuracy_tradeoff(0.6, Ms_t, Ns_t, ds_t)
assert res is not None
M_star, N_star, f_star = res
assert np.mean(1.0 - (1.0 - q_single(ds_t, M_star)) ** N_star) >= 0.6     # 达标
assert abs(f_star - infer_flops(M_star, N_star)) < 1e-6 * f_star          # FLOPs 口径一致
for M in Ms_t:                                                            # 暴力验证最优性
    for n in Ns_t:
        if np.mean(1.0 - (1.0 - q_single(ds_t, M)) ** n) >= 0.6:
            assert f_star <= infer_flops(M, n) * (1 + 1e-9)
assert iso_accuracy_tradeoff(0.999, Ms_t, Ns_t, ds_t) is None             # 不可行 → None
print("✅ 练习 3 通过")

## ✏️ 练习 4：实现 `loglog_fit(x, y)`

手写幂律拟合：对 $y \approx c \cdot x^p$ 形式的数据，在 log 域做一阶线性最小二乘，返回 `(slope, ln_c)`，即 $\ln y \approx \text{slope} \cdot \ln x + \ln c$。

**提示**：`np.polyfit(np.log(x), np.log(y), 1)` 一行搞定；输入要求 `x`、`y` 全为正。自测会：① 用真幂律数据验证参数精确恢复；② 用带乘性噪声的幂律验证稳健性；③ 用第 4 节的聚合 coverage 验证拟合斜率落在 $(0, 1)$。

In [ ]:
def loglog_fit(x, y):
    # TODO: log 域一阶最小二乘，返回 (slope, ln_c)，使 y ≈ exp(ln_c) * x**slope
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
x = np.logspace(0, 3, 20)
y = 3.0 * x ** (-0.7)
s, lc = loglog_fit(x, y)
assert abs(s - (-0.7)) < 1e-8                       # 真幂律：斜率精确恢复
assert abs(np.exp(lc) - 3.0) < 1e-8                 # 系数精确恢复
rng_t = np.random.default_rng(0)
y_noisy = y * np.exp(rng_t.normal(0.0, 0.05, size=len(x)))
s2, _ = loglog_fit(x, y_noisy)
assert abs(s2 - (-0.7)) < 0.1                       # 带噪仍稳健
s3, _ = loglog_fit(Ns_pl[seg_mask], cov_agg[seg_mask])
assert 0.0 < s3 < 1.0                               # 聚合 coverage 的幂律斜率 ∈ (0, 1)
print("✅ 练习 4 通过")

---
## 📖 参考答案

In [ ]:
# 练习 1 参考答案（先自己做，再对照）
def coverage_exact(q, N):
    return 1.0 - (1.0 - q) ** N

def coverage_relerr(q, N):
    exact = coverage_exact(q, N)
    return (q * N - exact) / exact

In [ ]:
# 练习 2 参考答案（先自己做，再对照）
def optimal_allocation(qs, B):
    qs = np.asarray(qs, dtype=float)
    alloc = np.zeros(len(qs), dtype=int)
    gains = qs.copy()                    # 每题下一条样本的边际收益 Δ_i = q_i (1-q_i)^{n_i}
    for _ in range(B):
        i = int(np.argmax(gains))
        alloc[i] += 1
        gains[i] *= (1.0 - qs[i])
    return alloc

In [ ]:
# 练习 3 参考答案（先自己做，再对照）
def iso_accuracy_tradeoff(target_acc, Ms, Ns, ds):
    feasible = []
    for M in Ms:
        qd = q_single(ds, M)
        for n in Ns:
            if np.mean(1.0 - (1.0 - qd) ** n) >= target_acc:
                feasible.append((infer_flops(M, n), M, n))
    if not feasible:
        return None
    f, M, n = min(feasible)
    return (M, n, f)

In [ ]:
# 练习 4 参考答案（先自己做，再对照）
def loglog_fit(x, y):
    slope, ln_c = np.polyfit(np.log(np.asarray(x, dtype=float)),
                             np.log(np.asarray(y, dtype=float)), 1)
    return slope, ln_c

---
## 小结

- 单题 coverage $1-(1-q)^N$ 是饱和曲线，但 $q_i$ 的重尾分布让**聚合** coverage 在 log-log 上近似线性——[Brown 2024] 的幂律，本实验拟合斜率 ≈ 0.13；
- TTC 收益**强烈依赖难度**：简单题几条样本就饱和、中等题持续受益、$q \approx 0$ 的超难题 $N$ 再大也救不了——只能加训练算力（[Snell 2024]）；
- compute-optimal 分配 = 贪心边际收益 $q(1-q)^n$：把预算从已饱和的简单题与无望的难题抽走、集中给中等题，期望解出数 +15% 左右（oracle 难度下的上界）；
- 在 $(\log M, \log N)$ 平面上：低/中目标 accuracy 的最省 FLOPs 组合落在"小模型+多采样"区域，高目标被迫滑向大模型——**不报推理预算的模型比较是不完整的陈述**；
- FLOPs 会计：一次前向 $\approx 2N_{\text{params}}\cdot$tokens、训练 $\approx 6ND$；"训大模型 vs 多采样"的总账由请求量（$R$ = 推理/训练 token 比）决定；
- 对评测的落点：**协议必须固定算力预算**——accuracy-per-FLOP 曲线取代单点 accuracy，pass@1 与 cons@64 不可混报。

下一章（06）研究 sequential scaling 的效率经济学：overthinking 与 budget forcing。

---
## 🎯 真实数据胶囊题：真实 pass@k 曲线的测试时计算 scaling

测试时多采样(更多计算)能提分，但边际递减。用真实 GSM8K 难度结构生成 pass@k 曲线，拟合 `acc ≈ a - b·exp(-c·k)` 式的饱和曲线，找“再加样本不划算”的拐点。

> 本模块新增的**真实数据**练习：自包含、用真实 GSM8K 把本章方法跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, urllib.request, re
import numpy as np
CACHE=os.path.expanduser("~/.reasoning_ttc_data"); os.makedirs(CACHE,exist_ok=True)
def _f(url,fn):
    p=os.path.join(CACHE,fn)
    if not os.path.exists(p): urllib.request.urlretrieve(url,p)
    return p
def gsm8k(n=300):
    p=_f("https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl","gsm8k_test.jsonl")
    return [json.loads(l) for l in open(p).read().splitlines()[:n]]
def gold(a): return a.split("####")[-1].strip().replace(",","")
def steps(a): return max(1, a.count("<<"))   # 真实推理步数代理

rows=gsm8k(200); ks_diff=np.array([steps(r["answer"]) for r in rows])
rng=np.random.default_rng(0)
p1=1/(1+np.exp((ks_diff-ks_diff.mean())/2.5))   # 真实难度->单次正确率
def pass_at_k(k):
    return float(np.mean(1-(1-p1)**k))           # 至少对一次
Ks=np.array([1,2,4,8,16,32,64])
accs=np.array([pass_at_k(k) for k in Ks])
print("真实 pass@k:", dict(zip(Ks.tolist(), accs.round(3).tolist())))

**练习**：实现 `marginal_gain(Ks, accs)`：返回每次“翻倍样本”带来的准确率增量数组。验证增量单调递减(边际递减)。

In [ ]:
def marginal_gain(Ks, accs):
    # TODO: 相邻 accs 的差(后-前)
    raise NotImplementedError


In [ ]:
# 自测
gains=marginal_gain(Ks, accs)
assert len(gains)==len(Ks)-1
assert (gains>=-1e-9).all(), "pass@k 单调不减 -> 增量非负"
assert gains[0] > gains[-1], "边际递减：后期翻倍收益更小"
print(f"翻倍样本的边际增益={gains.round(3)} ✓ 递减(测试时计算有性价比拐点)")


### 📖 参考答案

In [ ]:
def marginal_gain(Ks, accs):
    return np.diff(np.asarray(accs,float))
print("✓ 测试时计算 scaling 也遵循边际递减，要算 compute-optimal 分配")